In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import TargetEncoder

In [2]:
selected_months = pd.date_range(start='2025-02', end='2026-04', freq='MS').strftime('%Y%m')

listings = pd.concat([pd.read_csv(f'/Users/phillipphi/Desktop/IDX Exchange/Listing csv/CRMLSListing{month}.csv', low_memory=False) 
                      for month in selected_months], ignore_index=True)

sold = pd.concat([pd.read_csv(f'/Users/phillipphi/Desktop/IDX Exchange/Sold csv/CRMLSSold{month}.csv', low_memory=False) 
                      for month in selected_months], ignore_index=True)

In [3]:
listings = listings.loc[(listings["PropertyType"] == "Residential") & (listings["PropertySubType"] == "SingleFamilyResidence")]
listings.head()

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1
1,595500.0,1108426965,williamramsey760@gmail.com,NaN,NaN,William,Ramsey,34.183030,-116.074461,72772 Nandina Street,...,NaN,False,2.0,NaN,92277,NaN,NaN,99011.0,NaN,72772 Nandina Street
3,995000.0,1108426334,Jessica@native.us,NaN,NaN,Jessica,Foote,33.611316,-117.190621,26891 Aubrieta Street,...,1.0,False,4.0,Murrieta,92562,NaN,227.0,6970.0,NaN,26891 Aubrieta Street
4,1485000.0,1108425215,nickiwanderson@gmail.com,NaN,NaN,Nicki,Anderson,35.138586,-120.561044,608 Stagecoach,...,1.0,False,3.0,Lucia Mar Unified,93420,NaN,72.0,20473.2,NaN,608 Stagecoach
6,880000.0,1108423358,markchacon5@aol.com,NaN,NaN,Mark,Chacon,34.198774,-118.988364,6036 Gitana Avenue,...,NaN,NaN,2.0,NaN,93012,NaN,255.0,5227.0,NaN,6036 Gitana Avenue
11,1500000.0,1108422213,kim@kimfrankel.com,2025-03-08,1580000.0,Kimberly,Frankel,34.153178,-118.509516,4611 Louise Ave Avenue,...,1.0,False,2.0,Los Angeles Unified,91316,Keller Williams Beverly Hills,0.0,20473.2,NaN,4611 Louise Ave Avenue


In [4]:
sold = sold.loc[(sold["PropertyType"] == "Residential") & (sold["PropertySubType"] == "SingleFamilyResidence")]
sold.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,880000.0,497696903,lenskab@gmail.com,...,NaN,False,2.0,NaN,91942,NaN,NaN,NaN,NaN,NaN
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,875000.0,497696407,lenskab@gmail.com,...,NaN,False,2.0,NaN,91942,NaN,NaN,NaN,NaN,NaN
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,849000.0,486616176,lenskab@gmail.com,...,NaN,False,2.0,NaN,91942,NaN,NaN,NaN,NaN,NaN
15,SouthBay,SouthBay,Carpet,True,NaN,True,False,1100000.0,1108119618,elaine@elainemallon.com,...,4.0,False,2.0,Los Angeles Unified,90731,0.0,4501.0,NaN,NaN,NaN
16,ContraCosta,ContraCosta,Carpet,NaN,NaN,NaN,False,760000.0,1108119338,sabine.larsen@redfin.com,...,NaN,False,2.0,NaN,94579,NaN,5040.0,NaN,NaN,NaN


In [5]:
# remove duplicate columns from listing
dup_cols = listings.columns[listings.columns.str.endswith('.1')]
listings = listings.drop(columns = dup_cols)

listings.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency',
       'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor',
       'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal',
       'ContractStatusChangeDate', 'ElementarySchoolDistrict'

In [6]:
# count number of properties with a fireplace
# will use FireplaceYN instead of FireplacesTotal since FireplacesTotal is completely filled with NaN values

print(sold.FireplacesTotal.describe())
print(sold.FireplaceYN.describe())

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: FireplacesTotal, dtype: float64
count     162448
unique         2
top         True
freq      117794
Name: FireplaceYN, dtype: object


In [7]:
# only keeping important variables from listing

listings_var = ["ListingKey", "FireplaceYN", "AssociationFeeFrequency", "AttachedGarageYN", "ParkingTotal", 
                "YearBuilt", "BedroomsTotal", "Stories", "MainLevelBedrooms", "NewConstructionYN", "AssociationFee", 
                "LotSizeSquareFeet", "LivingArea", "City", "Latitude", "Longitude"]

listings = listings[listings_var]

In [8]:
sold.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey',
       'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName',
       'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea',
       'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',
       'ListingId', 'BathroomsTotalInteger', 'City', '

In [9]:
sold_var = ["ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN", "ListingKey", "ClosePrice", "FireplaceYN", 
            "AssociationFeeFrequency", "TaxAnnualAmount", "AttachedGarageYN", "ParkingTotal", 
            "YearBuilt", "BedroomsTotal", "Stories", "MainLevelBedrooms", "NewConstructionYN", 
            "AssociationFee", "LotSizeSquareFeet", "LivingArea", "City", "Latitude", "Longitude"]

sold = sold[sold_var]

In [10]:
# merge listings with sold data

merged = pd.merge(listings, sold, how='inner', on="ListingKey")
merged 

,ListingKey,FireplaceYN_x,AssociationFeeFrequency_x,AttachedGarageYN_x,ParkingTotal_x,YearBuilt_x,BedroomsTotal_x,Stories_x,MainLevelBedrooms_x,NewConstructionYN_x,...,BedroomsTotal_y,Stories_y,MainLevelBedrooms_y,NewConstructionYN_y,AssociationFee_y,LotSizeSquareFeet_y,LivingArea_y,City_y,Latitude_y,Longitude_y
0,1108425215,True,Monthly,True,3.0,2001.0,3.0,NaN,1.0,False,...,3.0,NaN,1.0,False,72.0,20473.2,2411.0,Arroyo Grande,35.138586,-120.561044
1,1108423358,True,Monthly,True,2.0,1985.0,2.0,1.0,NaN,NaN,...,2.0,1.0,NaN,NaN,265.0,5227.0,1561.0,Camarillo,34.198774,-118.988364
2,1108422213,True,NaN,True,2.0,1975.0,4.0,2.0,1.0,False,...,4.0,2.0,1.0,False,0.0,20473.2,3121.0,Encino,34.153178,-118.509516
3,1108420725,False,NaN,NaN,0.0,1924.0,2.0,1.0,2.0,False,...,2.0,1.0,2.0,False,0.0,3135.0,1274.0,West Hollywood,34.085973,-118.386947
4,1108409465,True,Monthly,True,2.0,1987.0,3.0,1.0,3.0,False,...,3.0,1.0,3.0,False,168.0,63597.6,1618.0,Coarsegold,37.179457,-119.768195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123920,1156016707,False,NaN,False,1.0,1897.0,3.0,1.0,NaN,False,...,3.0,1.0,NaN,False,NaN,3100.0,1320.0,Berkeley,37.867730,-122.293393
123921,1155791639,True,NaN,True,2.0,1968.0,3.0,1.0,NaN,False,...,3.0,1.0,NaN,False,NaN,6090.0,1675.0,San Jose,37.245049,-121.819955
123922,1155729135,True,Monthly,True,3.0,1985.0,5.0,2.0,NaN,False,...,5.0,2.0,NaN,False,210.0,20150.0,5383.0,Danville,37.812009,-121.897469
123923,1155356676,True,NaN,True,2.0,1948.0,7.0,1.0,NaN,False,...,7.0,1.0,NaN,False,NaN,18048.0,3035.0,Concord,37.953455,-122.002616


In [11]:
merged.columns

Index(['ListingKey', 'FireplaceYN_x', 'AssociationFeeFrequency_x',
       'AttachedGarageYN_x', 'ParkingTotal_x', 'YearBuilt_x',
       'BedroomsTotal_x', 'Stories_x', 'MainLevelBedrooms_x',
       'NewConstructionYN_x', 'AssociationFee_x', 'LotSizeSquareFeet_x',
       'LivingArea_x', 'City_x', 'Latitude_x', 'Longitude_x', 'ViewYN',
       'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'ClosePrice',
       'FireplaceYN_y', 'AssociationFeeFrequency_y', 'TaxAnnualAmount',
       'AttachedGarageYN_y', 'ParkingTotal_y', 'YearBuilt_y',
       'BedroomsTotal_y', 'Stories_y', 'MainLevelBedrooms_y',
       'NewConstructionYN_y', 'AssociationFee_y', 'LotSizeSquareFeet_y',
       'LivingArea_y', 'City_y', 'Latitude_y', 'Longitude_y'],
      dtype='object')

In [12]:
# some of the variables have duplicate names, except with _x or _y
# there are some inconsistencies with some of the values 
# we choose to trust _y more because it should be more updated
# if there are any NA values in the _y column, we will fill them with the corresponding _x value

y_dups = [col for col in merged.columns if col.endswith('_y')]

for col in y_dups:

    new_name = col[:-2]

    x_dup = col[:-1] + 'x'

    merged[new_name] = merged[col].fillna(merged[x_dup])

    merged = merged.drop(columns=[col, x_dup])

In [13]:
print(sum(merged.AssociationFee.isna()))
merged.AssociationFeeFrequency.unique()

38088


array(['Monthly', nan, 'Annually', 'Quarterly', 'SemiAnnually'],
      dtype=object)

In [14]:
# standardize the AssociationFee, we will turn it into yearly amounts

yearly_fee = []

row = 0

for fee_period in merged.AssociationFeeFrequency:
    fee = merged['AssociationFee'].iloc[row]

    if pd.isna(fee_period) or pd.isna(fee):
        yearly_fee.append(0)
    elif fee_period == 'Monthly':
        yearly_fee.append(fee * 12)
    elif fee_period == 'Quarterly':
        yearly_fee.append(fee * 4)
    elif fee_period == 'SemiAnnually':
        yearly_fee.append(fee * 2)
    elif fee_period == 'Annually':
        yearly_fee.append(fee)
    row += 1

merged['AssociationFeeYearly'] = yearly_fee
merged = merged.drop(columns=['AssociationFee', 'AssociationFeeFrequency'])

In [15]:
# for every column that contains simply a yes or no, we code them as 1 and 0, respectively
# assume na values are 0

yn_cols = [col for col in merged.columns if col.endswith('YN')]

for col in yn_cols:

    merged[col] = merged[col].fillna(0)

    merged[col] = merged[col].replace(to_replace = False, value = 0).infer_objects(copy=False)
    merged[col] = merged[col].replace(to_replace = True, value = 1).infer_objects(copy=False)

/var/folders/tx/vhbzq0d1553cy4n43fl_ct0w0000gn/T/ipykernel_9990/1348392797.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[col] = merged[col].replace(to_replace = True, value = 1).infer_objects(copy=False)


In [16]:
sold202506 = pd.read_csv(f'/Users/phillipphi/Desktop/IDX Exchange/Sold csv/CRMLSSold202506.csv', low_memory=False)

print(f"Proportion of missing TaxAnnualAmount values in sold202506: {sum(sold202506.TaxAnnualAmount.isna()) / len(sold202506)}")
print(f"Proportion of missing TaxAnnualAmount values in sold: {sum(sold.TaxAnnualAmount.isna()) / len(sold)}")

Proportion of missing TaxAnnualAmount values in sold202506: 0.9975090678669755
Proportion of missing TaxAnnualAmount values in sold: 1.0


In [17]:
# The proportion of missing TaxAnnualAmount in the dataset is extremely high
# Every value is missing, but it is likely due to the fact that it got removed after filtering for SingleFamilyResidence
# In the random sold csv I chose, over 99% of the values are missing
# Removing TaxAnnualAmount column 

merged = merged.drop(columns=['TaxAnnualAmount'])

In [18]:
# Fill missing values with 0
# These columns are most likely to have 0 values when missing

na_0_col = ["ParkingTotal", "BedroomsTotal", "MainLevelBedrooms"]

for col in na_0_col:
    merged[col] = merged[col].fillna(0)

In [19]:
# Remove rows with 0 BedroomsTotal since they are likely invalid entries
# unlikely that a Single Family Residence would have 0 bedrooms 

merged = merged[merged.BedroomsTotal != 0]

In [20]:
# Fill missing values with 1
# Missing stories are mostly likely to be 1

merged.loc[:, "Stories"] = merged["Stories"].fillna(1)

In [21]:
print(f"Number of missing ClosePrice values: {sum(merged.ClosePrice.isna())}")
print(f"Number of missing YearBuilt values: {sum(merged.YearBuilt.isna())}")
print(f"Number of missing LotSizeSquareFeet values: {sum(merged.LotSizeSquareFeet.isna())}")
print(f"Number of missing LivingArea values: {sum(merged.LivingArea.isna())}")
print(f"Number of missing Longitude values: {sum(merged.Longitude.isna())}")
print(f"Number of missing Latitude values: {sum(merged.Latitude.isna())}")

Number of missing ClosePrice values: 0
Number of missing YearBuilt values: 78
Number of missing LotSizeSquareFeet values: 2236
Number of missing LivingArea values: 60
Number of missing Longitude values: 171
Number of missing Latitude values: 171


In [22]:
# we will be dropping columns rows with missing values in YearBuilt, LotSizeSquareFeet, and LivingArea
# we will not be losing much data since it is only a small fraction of the dataset

drop_row_cols = ['YearBuilt', 'LotSizeSquareFeet', 'LivingArea', 'Longitude', 'Latitude']

merged = merged.dropna(subset=drop_row_cols)

In [23]:
# finds the column numbers of all numeric columns
# replaces negative values with positive values since they are most likely mis-inputs 

numeric_col = merged.select_dtypes(include=['number']).columns
numeric_noLatLong = numeric_col.difference(['Latitude', 'Longitude'])
merged.loc[:, numeric_noLatLong] = merged.loc[:, numeric_noLatLong].abs()

In [24]:
sold202604 = pd.read_csv('/Users/phillipphi/Desktop/IDX Exchange/Sold csv/CRMLSSold202604.csv')


# Finds the list of ListingKeys from the last month
keys_last_month = sold202604.ListingKey.tolist()

# uses the April 2026 data as the test set, which is the most recent data
# uses the remaining data as the training set
test_data = merged[merged.ListingKey.isin(keys_last_month)]
training_data = merged[~merged.ListingKey.isin(keys_last_month)]

test_data = test_data.drop(columns=['ListingKey'])
training_data = training_data.drop(columns=['ListingKey'])

In [25]:
# removes extremely small and large values that are likely outliers caused by mis-inputs

numeric_col = training_data.select_dtypes(include=['number']).columns

training_data.loc[:, numeric_col] = \
    training_data.loc[:, numeric_col].clip(lower=training_data.loc[:, numeric_col].quantile(0.01), 
                                           upper=training_data.loc[:, numeric_col].quantile(0.99), axis=1)

In [26]:
for col in training_data.columns:
    print(training_data[col].describe())

count    111204.000000
mean          0.554171
std           0.497059
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: ViewYN, dtype: float64
count    111204.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: WaterfrontYN, dtype: float64
count    111204.000000
mean          0.024001
std           0.153053
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: BasementYN, dtype: float64
count    111204.000000
mean          0.151227
std           0.358271
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: PoolPrivateYN, dtype: float64
count    1.112040e+05
mean     1.203398e+06
std      9.724246e+05
min      2.400000e+05
25%      6.280000e+05
50%      8.920000e+05
75%      1.415000e+06
max      6.189550

In [27]:
# Our above description of variables shows that WaterfrontYN is entirely 0
# it most likely got removed after filtering out outliers
# in our sold dataset, less than 0.1% of the properties have a waterfront, so it will likely not be useful as a factor
# we will remove it from the training and test data 

print(training_data.WaterfrontYN.describe()) 
print(round(sold.WaterfrontYN.sum() / len(sold), 4))

training_data = training_data.drop(columns=['WaterfrontYN'])
test_data = test_data.drop(columns=['WaterfrontYN'])

count    111204.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: WaterfrontYN, dtype: float64
0.0005


In [28]:
merged.City.value_counts()[1:150]

City
San Diego        3596
San Jose         1909
Riverside        1862
Oakland          1745
Menifee          1362
                 ... 
Van Nuys          236
Sunnyvale         234
Buena Park        234
Claremont         233
Anaheim Hills     233
Name: count, Length: 149, dtype: int64

In [29]:
# performs target encoding on the city feature

encode = TargetEncoder(smooth="auto", target_type="continuous")

training_data["City"] = encode.fit_transform(training_data[["City"]], training_data["ClosePrice"])
test_data["City"] = encode.transform(test_data[["City"]])

In [30]:
training_data.to_csv('training_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)